<div align="center">
  <hr>
  <h1>Análise de Teste A/B: Avaliação do Novo Sistema de Recomendação</h1>
  
  <hr>
</div>

### Caso

Uma loja online internacional implementou um **novo sistema de recomendação de produtos** com o objetivo de melhorar a experiência do usuário e aumentar as taxas de conversão. Para medir sua eficácia, foi conduzido um teste A/B entre 7 e 21 de dezembro de 2020. Os novos usuários foram divididos em dois grupos:
*   **Grupo A (Controle):** Interagiu com o sistema antigo.
*   **Grupo B (Teste):** Interagiu com o novo sistema de recomendação.

### O Desafio

Apesar do planejamento, os resultados iniciais do teste foram considerados **inconclusos** e a análise foi interrompida. Surgiram suspeitas sobre a integridade da coleta de dados e a configuração do experimento, como o público-alvo, o tamanho da amostra e o balanceamento dos grupos.


## Sumário
1.  [**Objetivos**](#Objetivos)
2.  [**Análise e Preparação dos Dados**](#Análise-e-Preparação-dos-Dados)
    *   [2.1. Pré-processamento e Limpeza](#Pré-processamento-e-Limpeza-dos-Dados)
    *   [2.2. Verificação de Consistência do Teste](#Verificação-de-consistência-do-Teste)
    *   [2.3. Análise Exploratória de Dados (AED)](#Análise-Exploratória-de-Dados-(AED))
    *   [2.4. Análise e Tratamento de Outliers](#Análise-e-Tratamento-de-Outliers)
3.  [**Análise do Teste A/B**](#Análise-do-Teste-A/B)
    *   [3.1. Preparação dos Dados](#Preparando-dados-para-o-teste)
    *   [3.2. Execução do Teste Z](#Teste-A/B-1)
    *   [3.3. Interpretação dos Resultados](#Interpretação-dos-Resultados)
4.  [**Conclusão Final e Recomendações**](#Conclusão-Final)
    *   [4.1. Recomendações](#Recomendações)

## Objetivos

O objetivo principal deste projeto é **avaliar a eficácia do novo sistema de recomendação** implementado na loja online, através da análise de um teste A/B inconcluso.

Especificamente, a análise visa:

1.  **Verificar a conformidade da execução do teste A/B** com as especificações técnicas fornecidas, identificando e documentando quaisquer inconsistências nos dados ou na configuração do teste (como público, período, etc.) que possam impactar a validade dos resultados.
2.  **Analisar o funil de conversão** (`product_page` → `product_cart` → `purchase`) para os usuários participantes do teste (Grupos A e B).
3.  **Determinar se o novo sistema de recomendação (Grupo B)** gerou um aumento estatisticamente significativo nas taxas de conversão, comparado ao grupo de controle (Grupo A), em cada etapa do funil (`product_page`, `product_cart`, `purchase`) dentro de 14 dias após o cadastro do usuário.
4.  **Avaliar se o resultado alcançado (Grupo B vs Grupo A)** cumpre o critério de sucesso esperado: um aumento de **pelo menos 10%** na conversão em cada uma das três etapas do funil.
5.  **Fornecer conclusões e recomendações** sobre a implementação ou não do novo sistema de recomendação, considerando tanto os resultados estatísticos quanto quaisquer problemas identificados na execução do teste.

In [1]:
import pandas as pd, plotly.graph_objects as go, plotly.express as px, os
from statsmodels.stats.proportion import proportions_ztest
import warnings

### Datasets

##### 1. Calendário de Eventos de Marketing (`ab_project_marketing_events_us.csv`)

Este arquivo é o nosso mapa dos eventos externos que podem ter influenciado a chegada e o comportamento dos usuários. Ele nos mostra quais campanhas de marketing estavam ativas e onde, fornecendo contexto importante para a análise.

*   **`name`**: O nome da campanha ou evento de marketing. Uma etiqueta clara para identificar a iniciativa.
*   **`regions`**: As regiões geográficas (separadas por vírgula) onde a campanha foi veiculada. Essencial para cruzar com a localização dos usuários.
*   **`start_dt`**: A data de início da campanha. O ponto de partida da ação de marketing.
*   **`finish_dt`**: A data de término da campanha. O encerramento da iniciativa de marketing.

##### 2. Novos Usuários Cadastrados (`final_ab_new_users_upd_us.csv`)

Este dataset é o ponto de partida da nossa população em estudo: todos os usuários recém-chegados que se registraram na loja online durante o período de 7 a 21 de dezembro de 2020.

*   **`user_id`**: Um identificador único para cada novo usuário. A chave para rastrear o indivíduo.
*   **`first_date`**: A data exata em que o usuário completou seu cadastro na plataforma. O momento de "nascimento" digital do usuário na loja.
*   **`region`**: A região geográfica associada ao usuário. Ajuda a segmentar a análise por localização.
*   **`device`**: O tipo de dispositivo (mobile, desktop, etc.) que o usuário utilizou para se cadastrar. Crucial para entender a jornada de acesso.

##### 3. Eventos e Comportamento dos Usuários (`final_ab_events_upd_us.csv`)

O coração da análise comportamental. Este arquivo registra cada ação significativa (eventos) realizada pelos *novos usuários* (aqueles do dataset anterior) entre 7 de dezembro de 2020 e 1º de janeiro de 2021.

*   **`user_id`**: Identificador do usuário que realizou o evento. Liga a ação ao indivíduo.
*   **`event_dt`**: O timestamp (data e hora) exato em que o evento ocorreu. Permite analisar a sequência e o timing das ações.
*   **`event_name`**: O nome do tipo de evento que aconteceu. Pode ser um login, visualização de produto, adição ao carrinho, compra, etc. O "o que" da ação.
*   **`details`**: Dados adicionais e contextuais sobre o evento. Por exemplo, para um evento de `purchase`, este campo pode conter o valor total da compra em USD. Fornece detalhes qualitativos sobre a ação.

##### 4. Participantes do Teste A/B (`final_ab_participants_upd_us.csv`)

Esta tabela é a peça-chave para segmentar os usuários nos grupos do experimento. Ela nos diz quem, entre os novos usuários, foi incluído no teste A/B e a qual grupo (Controle ou Teste) essa pessoa foi atribuída.

*   **`user_id`**: Identificador do usuário que participa do teste. Conecta o indivíduo ao experimento.
*   **`ab_test`**: O nome ou identificador específico do teste A/B em questão. Útil se houver múltiplos testes sendo executados simultaneamente (embora neste caso provavelmente seja apenas um principal).
*   **`group`**: O grupo ao qual o usuário foi atribuído dentro do teste A/B (e.g., 'A' para Controle, 'B' para Teste). Define qual versão da experiência o usuário viu.



In [2]:
servidor = '' 
caminho_arquivo = os.path.join(servidor, 'datasets') # mude para servidor caso for no jupter

df_marketing = pd.read_csv(os.path.join(caminho_arquivo, 'ab_project_marketing_events_us.csv'))
df_events = pd.read_csv(os.path.join(caminho_arquivo, 'final_ab_events_upd_us.csv'))
df_new_users = pd.read_csv(os.path.join(caminho_arquivo, 'final_ab_new_users_upd_us.csv'))
df_participants = pd.read_csv(os.path.join(caminho_arquivo, 'final_ab_participants_upd_us.csv'))

### Pré-processamento e Limpeza dos Dados

In [3]:
df_marketing.info()
print()
df_marketing['start_dt'] = pd.to_datetime(df_marketing['start_dt'], errors='coerce')
df_marketing['finish_dt'] = pd.to_datetime(df_marketing['finish_dt'], errors='coerce')
df_marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   name       14 non-null     object
 1   regions    14 non-null     object
 2   start_dt   14 non-null     object
 3   finish_dt  14 non-null     object
dtypes: object(4)
memory usage: 580.0+ bytes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   name       14 non-null     object        
 1   regions    14 non-null     object        
 2   start_dt   14 non-null     datetime64[ns]
 3   finish_dt  14 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 580.0+ bytes


In [4]:
df_events.info()
print()
df_events['event_name'] = df_events['event_name'].astype('category')
df_events['event_dt'] = pd.to_datetime(df_events['event_dt'], errors='coerce')
df_events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 423761 entries, 0 to 423760
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   user_id     423761 non-null  object 
 1   event_dt    423761 non-null  object 
 2   event_name  423761 non-null  object 
 3   details     60314 non-null   float64
dtypes: float64(1), object(3)
memory usage: 12.9+ MB

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 423761 entries, 0 to 423760
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   user_id     423761 non-null  object        
 1   event_dt    423761 non-null  datetime64[ns]
 2   event_name  423761 non-null  category      
 3   details     60314 non-null   float64       
dtypes: category(1), datetime64[ns](1), float64(1), object(1)
memory usage: 10.1+ MB


In [5]:
df_new_users.info()
print()
df_new_users['region'] = df_new_users['region'].astype('category')
df_new_users['device'] = df_new_users['device'].astype('category')
df_new_users['first_date'] = pd.to_datetime(df_new_users['first_date'], errors='coerce')
df_new_users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58703 entries, 0 to 58702
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_id     58703 non-null  object
 1   first_date  58703 non-null  object
 2   region      58703 non-null  object
 3   device      58703 non-null  object
dtypes: object(4)
memory usage: 1.8+ MB

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58703 entries, 0 to 58702
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   user_id     58703 non-null  object        
 1   first_date  58703 non-null  datetime64[ns]
 2   region      58703 non-null  category      
 3   device      58703 non-null  category      
dtypes: category(2), datetime64[ns](1), object(1)
memory usage: 1.0+ MB


In [6]:
df_participants.info()
print()
df_participants['group'] = df_participants['group'].astype('category')
df_participants['ab_test'] = df_participants['ab_test'].astype('category')
df_participants.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14525 entries, 0 to 14524
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  14525 non-null  object
 1   group    14525 non-null  object
 2   ab_test  14525 non-null  object
dtypes: object(3)
memory usage: 340.6+ KB

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14525 entries, 0 to 14524
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   user_id  14525 non-null  object  
 1   group    14525 non-null  category
 2   ab_test  14525 non-null  category
dtypes: category(2), object(1)
memory usage: 142.2+ KB


In [7]:
df_participants['ab_test'].value_counts()

ab_test
interface_eu_test          10850
recommender_system_test     3675
Name: count, dtype: int64

### Types
*   **Datas (`object` para `datetime`):**
    *   `start_dt` e `finish_dt` no DataFrame de informações do teste 
    *   `event_dt` no DataFrame de eventos 
    *   `first_date` no DataFrame de usuários 

*   **Categorias (`object` para `category`):**
    *   `event_name` no DataFrame de eventos 
    *   `region` e `device` no DataFrame de usuários 
    *   `group` e `ab_test` no DataFrame do teste 

Resumo do Impacto no Tamanho:

 - DataFrame de Eventos: Reduzido de 12.9 MB para 10.1 MB.
 - DataFrame de Usuários: Reduzido de 1.8 MB para 1.0 MB (quase 50% de economia).
 - DataFrame do Teste: Reduzido de 340.6 KB para 142.2 KB (mais de 50% de economia).'

### Valores ausentes ou duplicados.

In [8]:
df_events

,user_id,event_dt,event_name,details
0,E1BDDCE0DAFA2679,2020-12-07 20:22:03,purchase,99.99
1,7B6452F081F49504,2020-12-07 09:22:53,purchase,9.99
2,9CD9F34546DF254C,2020-12-07 12:59:29,purchase,4.99
3,96F27A054B191457,2020-12-07 04:02:40,purchase,4.99
4,1FD7660FDF94CA1F,2020-12-07 10:15:09,purchase,4.99
...,...,...,...,...
423756,245E85F65C358E08,2020-12-30 19:35:55,login,NaN
423757,9385A108F5A0A7A7,2020-12-30 10:54:15,login,NaN
423758,DB650B7559AC6EAC,2020-12-30 10:59:09,login,NaN
423759,F80C9BDDEA02E53C,2020-12-30 09:53:39,login,NaN


In [9]:
def ausentes(list):
    """
    Verifica e imprime a contagem de linhas completamente duplicadas para cada DataFrame em uma lista.
    """
    for i in list:
        print(i.isnull().sum())

list_df = [df_marketing, df_events, df_new_users, df_participants]

ausentes(list_df)

name         0
regions      0
start_dt     0
finish_dt    0
dtype: int64
user_id            0
event_dt           0
event_name         0
details       363447
dtype: int64
user_id       0
first_date    0
region        0
device        0
dtype: int64
user_id    0
group      0
ab_test    0
dtype: int64


Existem dados nulos no `df_events`, porém é importantes que se mantenham assim. 

In [10]:
def duplicates(list):
    """
    Verifica e imprime a contagem de linhas completamente duplicadas para cada DataFrame em uma lista.
    """
    for i in list:
        print(i.duplicated().sum())

list_df = [df_marketing, df_events, df_new_users, df_participants]

duplicates(list_df)

0
0
0
0


In [11]:
# O user_id deve ser único para cada novo usuário
print(f"df_new_users:{df_new_users.duplicated(subset=['user_id']).sum()}")
# A combinação user_id e first_date também deve ser única
print(f"df_new_users:{df_new_users.duplicated(subset=['user_id', 'first_date']).sum()}")
# Verificando duplicados na combinação principal de eventos (user_id, event_dt, event_name)
print(f"df_events:{df_events.duplicated(subset=['user_id', 'event_dt', 'event_name']).sum()}")
# O user_id deve ser único - CRÍTICO para atribuição de grupo
print(f"df_participants:{df_participants.duplicated(subset=['user_id']).sum()}")

df_new_users:0
df_new_users:0
df_events:0
df_participants:887


In [12]:
groups = df_participants.groupby('user_id')['ab_test'].nunique() == 1
groups[groups].index

Index(['0002CE61FF2C4011', '0010A1C096941592', '001E72F50D1C48FA',
       '002412F1EB3F6E38', '002540BE89C930FB', '0031F1B5E9FBF708',
       '003346BB64227D0C', '0036BE15EE4D319D', '003DF44D7589BBD4',
       '003F86A34B575D27',
       ...
       'FFC676CB3E0A60B8', 'FFD58017F5FA2DAC', 'FFDA3BD9A090A179',
       'FFDC1BEFD27A66D5', 'FFE40BDB7364E966', 'FFE5B14BD55C1C5C',
       'FFE600EEC4BA7685', 'FFE7FC140521F5F6', 'FFEFC0E55C1CCD4F',
       'FFF58BC33966EB51'],
      dtype='object', name='user_id', length=12751)

In [13]:
df_participants = df_participants.drop_duplicates(subset=['user_id'], keep='first')
print(f"df_participants: {df_participants.duplicated(subset=['user_id']).sum()}")

df_participants: 0


#### Removi todos os duplicados explicitos

In [14]:
users_in_multiple_tests_overall = df_participants.groupby('user_id')['ab_test'].nunique()
count_multiple_tests_users = len(users_in_multiple_tests_overall[users_in_multiple_tests_overall > 1])
print(count_multiple_tests_users)

0


Não tem nenhum usuário em mais de 1 teste.

### Verificação de consistência do Teste 

#### Discrepância entre Objetivo e Dados

O objetivo do teste, conforme a descrição técnica, era avaliar o sistema de recomendação na **região da UE**. No entanto, os nomes dos arquivos (`*_us.csv`) e o conteúdo dos dados indicam que a coleta foi realizada nos **Estados Unidos (US)**.

Esta é uma inconsistência crítica que invalida a aplicação direta dos resultados para a UE. A análise prosseguirá utilizando os dados disponíveis dos EUA, e essa limitação será destacada nas conclusões.

Regiões presentes nos dados:

In [15]:
df_new_users['region'].unique()

['EU', 'N.America', 'APAC', 'CIS']
Categories (4, object): ['APAC', 'CIS', 'EU', 'N.America']

In [16]:
df_new_users = df_new_users[(df_new_users['region'] == 'EU')]

In [17]:
# Garante que os participantes do teste também pertencem ao grupo de usuários válidos 
# (ou seja, da região 'EU')
valid_user_ids = df_new_users['user_id'].unique()
df_participants = df_participants[df_participants['user_id'].isin(valid_user_ids)]

Filtro de datas

In [18]:
start_date = '2020-12-07'
end_date = '2020-12-21'
df_new_users = df_new_users[(df_new_users['first_date'] >= start_date) & (df_new_users['first_date'] <= end_date)]

In [19]:
# Define a data limite para os eventos
event_end_date = '2021-01-01'

# Filtra eventos pela data limite
df_events = df_events[df_events['event_dt'] <= event_end_date]

# Obtém a lista de user_ids dos usuários que se cadastraram no período correto
# (df_new_users já foi filtrado no passo anterior)
valid_user_ids = df_new_users['user_id'].unique()

# Filtra eventos para incluir apenas aqueles realizados por usuários válidos
df_events = df_events[df_events['user_id'].isin(valid_user_ids)]

print(f"Número de eventos após filtrar por data e usuários: {len(df_events)}")

Número de eventos após filtrar por data e usuários: 292932


In [20]:
# Junta os eventos com as datas de primeiro registro dos usuários
df_events = df_events.merge(df_new_users[['user_id', 'first_date']], on='user_id', how='left')

# Calcula a diferença em dias entre o evento e a data de registro
df_events['days_since_registration'] = (df_events['event_dt'] - df_events['first_date']).dt.days

# Filtra eventos que ocorreram em até 14 dias
df_events = df_events[df_events['days_since_registration'] <= 14]

# Remove as colunas temporárias de data de registro e dias
df_events = df_events.drop(columns=['first_date', 'days_since_registration'])

print(f"Número de eventos após filtrar por 14 dias pós-cadastro: {len(df_events)}")

Número de eventos após filtrar por 14 dias pós-cadastro: 282869


In [21]:
test_name = 'recommender_system_test'
df_participants = df_participants[df_participants['ab_test'] == test_name]

print(f"Participantes restantes após filtrar por teste '{test_name}': {len(df_participants)}")

Participantes restantes após filtrar por teste 'recommender_system_test': 3481


In [22]:
# O DataFrame df_participants já foi limpo e filtrado para o teste específico
actual_participants_count = len(df_participants['user_id'].unique())
expected_participants_count = 6000

print(f"Número real de participantes no teste '{df_participants['ab_test'].iloc[0]}': {actual_participants_count}")
print(f"Número esperado de participantes: {expected_participants_count}")

Número real de participantes no teste 'recommender_system_test': 3481
Número esperado de participantes: 6000


Ter 3675 participantes não é o ideal para o teste, pois está abaixo do planejado e pode comprometer a capacidade de detectar o efeito desejado (aumento de 10%).

In [23]:
unique_users_count = df_participants['user_id'].nunique()
len(df_participants)

3481

Todos os usuários estão em apenas um grupo para o teste 'recommender_system_test'.

### Análise Exploratória de Dados (AED)

In [24]:
print(f"""Usuarios:
      grupo A: {df_participants[df_participants['group'] == "A"]['group'].count()}
      grupo B: {df_participants['group'].count()-df_participants[df_participants['group'] == "A"]['group'].count()}""")

Usuarios:
      grupo A: 2604
      grupo B: 877


Esta é mais uma inconsistência crítica na forma como o teste foi executado. Sugere que o mecanismo de atribuição de grupo não funcionou corretamente, pois não dividiu os usuários de forma equitativa entre os dois grupos.

In [25]:
# Junta info de região e dispositivo aos participantes
df_participants_details = df_participants.merge(
    df_new_users[['user_id', 'region', 'device']],
    on='user_id',
    how='left'
)

# Prepara os dados para os gráficos
region_dist = df_participants_details.groupby(['group', 'region']).size().reset_index(name='count')
device_dist = df_participants_details.groupby(['group', 'device']).size().reset_index(name='count')

# Gráfico 1: Distribuição por Região e Grupo
fig_region = px.bar(
    region_dist,
    x='region',
    y='count',
    color='group',
    barmode='group',
    title='Distribuição de Usuários por Região e Grupo',
    labels={'count': 'Número de Usuários', 'region': 'Região', 'group': 'Grupo'},
    template='plotly_dark'
)
fig_region.show()

# Gráfico 2: Distribuição por Dispositivo e Grupo
fig_device = px.bar(
    device_dist,
    x='device',
    y='count',
    color='group',
    barmode='group',
    title='Distribuição de Usuários por Dispositivo e Grupo',
    labels={'count': 'Número de Usuários', 'device': 'Dispositivo', 'group': 'Grupo'},
    template='plotly_dark'
)
fig_device.show()

C:\Users\jonat\AppData\Local\Temp\ipykernel_21300\2305536777.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  region_dist = df_participants_details.groupby(['group', 'region']).size().reset_index(name='count')
C:\Users\jonat\AppData\Local\Temp\ipykernel_21300\2305536777.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  device_dist = df_participants_details.groupby(['group', 'device']).size().reset_index(name='count')


Olhando para a proporção de usuários por região e por dispositivo dentro de cada grupo, a distribuição parece bem equilibrada. Isso sugere que a randomização, apesar de ter criado grupos de tamanhos muito diferentes (A=2747, B=928), distribuiu bem as características demográficas (região, dispositivo) entre eles.

In [26]:
users_login = df_events[df_events['event_name'] == 'login']['user_id'].nunique()
users_page = df_events[df_events['event_name'] == 'product_page']['user_id'].nunique()
users_cart = df_events[df_events['event_name'] == 'product_cart']['user_id'].nunique()
users_purchase = df_events[df_events['event_name'] == 'purchase']['user_id'].nunique()

stages = ['Login', 'Visualizou Página de Produto', 'Adicionou ao Carrinho', 'Comprou']
values = [users_login, users_page, users_cart, users_purchase]

fig = go.Figure(go.Funnel(
    y=stages,
    x=values,
    textinfo="value+percent initial",
    opacity=0.8,
    marker={"color": ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"],
            "line": {"width": 2, "color": "#2d2d2d"}},
    connector={"line": {"color": "#636efa", "dash": "dot", "width": 3}}
))

fig.update_layout(
    title_text='Funil de Conversão (Login -> Visualização -> Carrinho -> Compra)',
    title_x=0.5,
    template='plotly_dark'
)

fig.show()

 O número de compras (13k) é ligeiramente maior que o de adição ao carrinho (12k), o que é incomum e sugere um desvio no fluxo padrão ou no registro dos eventos.
 Porém o motivo desve ser por compras diretas. 

In [27]:
if users_purchase > users_cart:
    print(f"Observação: {users_purchase} usuários únicos compraram, mas apenas {users_cart} adicionaram ao carrinho.")
    print("Isso sugere que alguns usuários estão comprando sem passar pelo evento 'Adicionar ao Carrinho'.")
else:
    print(f"Consistente: {users_purchase} usuários únicos compraram (<= {users_cart} que adicionaram ao carrinho).")

Observação: 13748 usuários únicos compraram, mas apenas 12961 adicionaram ao carrinho.
Isso sugere que alguns usuários estão comprando sem passar pelo evento 'Adicionar ao Carrinho'.


In [28]:
# Extrair a data (sem hora) do timestamp do evento
df_events['event_date'] = df_events['event_dt'].dt.date

# Contar o número de eventos por data
events_per_day = df_events['event_date'].value_counts().sort_index()

# Converter para DataFrame para Plotly Express
events_per_day_df = events_per_day.reset_index()
events_per_day_df.columns = ['date', 'event_count']
events_per_day_df['date'] = pd.to_datetime(events_per_day_df['date']) # Convert back to datetime for plotting

# Criar o gráfico de linha
fig = px.bar(
    events_per_day_df,
    x='date',
    y='event_count',
    title='Número Total de Eventos por Dia',
    labels={'date': 'Data', 'event_count': 'Número de Eventos'},
)

# Ajustes de layout para tema escuro e melhor visualização
fig.update_layout(
    template='plotly_dark',
    xaxis_title='Data',
    yaxis_title='Número de Eventos',
    hovermode='x unified' # Mostrar informações de todos os traços para a mesma data
)

fig.show()

O número total de eventos varia diariamente, com um pico notável por volta de 21 de dezembro. Após essa data, a atividade de eventos diminui consideravelmente. Isso sugere que a maioria da atividade registrada ocorreu durante o período de recrutamento de usuários e/ou dentro da janela de 14 dias pós-cadastro.

In [29]:
# Calcular o número total de eventos por dia
events_by_day = df_events.groupby(df_events['event_dt'].dt.date).size().reset_index(name='event_count')
events_by_day['event_dt'] = pd.to_datetime(events_by_day['event_dt'])

# Criar o gráfico base de eventos por dia
fig = go.Figure(data=[go.Bar(
    x=events_by_day['event_dt'],
    y=events_by_day['event_count']
)])

# Adicionar datas dos eventos de marketing ao gráfico
shapes = []
annotations = []
for index, row in df_marketing.iterrows():
    # Adiciona uma forma (retângulo) para o período do evento
    shapes.append(dict(
        type="rect",
        xref="x",
        yref="paper",
        x0=row['start_dt'],
        y0=0,
        x1=row['finish_dt'],
        y1=1,
        fillcolor="rgba(255, 0, 0, 0.1)", # Cor semi-transparente (vermelho fraco)
        line_width=0,
        layer="below"
    ))
    # Adiciona uma anotação para o nome do evento
    annotations.append(dict(
        x=row['start_dt'], # Posição X na data de início do evento
        y=1.05,           # Posição Y (acima do gráfico)
        xref="x",
        yref="paper",
        text=row['name'],
        showarrow=False,
        xanchor='left',
        font=dict(size=10, color="red")
    ))


# Atualizar layout do gráfico com as formas e anotações
fig.update_layout(
    title_text='Número Total de Eventos por Dia com Eventos de Marketing',
    title_x=0.5,
    xaxis_title="Data",
    yaxis_title="Número de Eventos",
    shapes=shapes,
    annotations=annotations,
    template='plotly_dark'
)

fig.show()


As campanhas de marketing podem ter influenciado nos grupos, mesmo que seja igualmente é importante destacar. 

In [30]:
# Junta eventos com a informação do grupo
df_events_grouped = df_events.merge(df_participants[['user_id', 'group']], on='user_id', how='inner')

# Conta eventos por dia e por grupo
events_by_day_group = df_events_grouped.groupby([df_events_grouped['event_dt'].dt.date, 'group']).size().unstack(fill_value=0)
events_by_day_group.index = pd.to_datetime(events_by_day_group.index)

# Cria o gráfico
fig = go.Figure()

fig.add_trace(go.Bar(
    x=events_by_day_group.index,
    y=events_by_day_group['A'],
    name='Grupo A'
))

fig.add_trace(go.Bar(
    x=events_by_day_group.index,
    y=events_by_day_group['B'],
    name='Grupo B'
))

fig.update_layout(
    title_text='Número Total de Eventos Diários por Grupo (A vs B)',
    title_x=0.5,
    xaxis_title="Data",
    yaxis_title="Número de Eventos",
    template='plotly_dark'
)

fig.show()

C:\Users\jonat\AppData\Local\Temp\ipykernel_21300\2898135350.py:5: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [31]:

events_by_day_group.columns = events_by_day_group.columns.astype(str)

# Encontrar o valor máximo de eventos para cada grupo separadamente
peak_events_A = events_by_day_group['A'].max()
peak_events_B = events_by_day_group['B'].max()

# Calcular a porcentagem de eventos diários para cada grupo em relação AO SEU pico
events_by_day_group['percent_A_relative_peak'] = (events_by_day_group['A'] / peak_events_A) * 100 if peak_events_A > 0 else 0
events_by_day_group['percent_B_relative_peak'] = (events_by_day_group['B'] / peak_events_B) * 100 if peak_events_B > 0 else 0


# Criar o gráfico de linhas com as porcentagens relativas
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=events_by_day_group.index,
    y=events_by_day_group['percent_A_relative_peak'],
    mode='lines+markers',
    name='Grupo A (% do seu Pico Diário)'
))

fig.add_trace(go.Scatter(
    x=events_by_day_group.index,
    y=events_by_day_group['percent_B_relative_peak'],
    mode='lines+markers',
    name='Grupo B (% do seu Pico Diário)'
))

fig.update_layout(
    title_text='Atividade Diária por Grupo (% do Pico Diário de Cada Grupo)',
    title_x=0.5,
    xaxis_title="Data",
    yaxis_title="Atividade Diária (% do Pico do Grupo)",
    template='plotly_dark'
)

fig.show()

A diferença no gráfico de eventos diários se deve principalmente ao tamanho desigual dos grupos.

### Análise e Tratamento de Outliers

In [32]:
orders_per_user = df_events[df_events['event_name'] == 'purchase'].groupby('user_id')['event_name'].count()

fig = px.scatter(
    y=orders_per_user,
    opacity=0.09,
    title='Dispersão do Número de Pedidos por Usuário',
    labels={'y': 'Número de Pedidos', 'index': 'Usuário (índice)'}
)

fig.update_layout(template='plotly_dark', title_x=0.5)
fig.show()

print(orders_per_user.quantile([0.95, 0.99]))

0.95    5.0
0.99    6.0
Name: event_name, dtype: float64


In [33]:
anomalous_user_ids = orders_per_user[orders_per_user > 5].index
df_participants_clean = df_participants[~df_participants['user_id'].isin(anomalous_user_ids)]
df_events = df_events[~df_events['user_id'].isin(anomalous_user_ids)]

In [34]:
print(orders_per_user.quantile([0.95, 0.99]))

0.95    5.0
0.99    6.0
Name: event_name, dtype: float64


Removi os dados dos usuários com um numero grande de compras. 

### Teste A/B

#### Preparando dados para o teste

In [35]:
# Primeiro, pegamos a lista de usuários que estão no Grupo A
group_a_users = df_participants[df_participants['group'] == 'A']['user_id']

# Agora, contamos quantos desses usuários únicos aparecem em cada evento
success_page_a = df_events[(df_events['event_name'] == 'product_page') & (df_events['user_id'].isin(group_a_users))]['user_id'].nunique()
success_cart_a = df_events[(df_events['event_name'] == 'product_cart') & (df_events['user_id'].isin(group_a_users))]['user_id'].nunique()
success_purchase_a = df_events[(df_events['event_name'] == 'purchase') & (df_events['user_id'].isin(group_a_users))]['user_id'].nunique()

# Primeiro, pegamos a lista de usuários que estão no Grupo B
group_b_users = df_participants[df_participants['group'] == 'B']['user_id']

# Agora, contamos quantos desses usuários únicos aparecem em cada evento
success_page_b = df_events[(df_events['event_name'] == 'product_page') & (df_events['user_id'].isin(group_b_users))]['user_id'].nunique()
success_cart_b = df_events[(df_events['event_name'] == 'product_cart') & (df_events['user_id'].isin(group_b_users))]['user_id'].nunique()
success_purchase_b = df_events[(df_events['event_name'] == 'purchase') & (df_events['user_id'].isin(group_b_users))]['user_id'].nunique()

# Dados de sucesso e totais (assumindo que já foram calculados)
sucessos_a = [success_page_a, success_cart_a, success_purchase_a]
sucessos_b = [success_page_b, success_cart_b, success_purchase_b]
total_a = df_participants[df_participants['group'] == 'A']['user_id'].nunique()
total_b = df_participants[df_participants['group'] == 'B']['user_id'].nunique()
# Nomes das etapas
etapas = ['product_page', 'product_cart', 'purchase']

# Criar o dicionário com os dados
data = {
    'Etapa': etapas + etapas,
    'Grupo': ['A'] * 3 + ['B'] * 3,
    'Quantidade (Sucessos)': sucessos_a + sucessos_b,
    'Total (Tentativas)': [total_a] * 3 + [total_b] * 3
}

# Criar o DataFrame
summary_df = pd.DataFrame(data)

# Calcular a coluna de porcentagem
summary_df['Porcentagem (%)'] = (summary_df['Quantidade (Sucessos)'] / summary_df['Total (Tentativas)']) * 100
summary_df['Porcentagem (%)'] = summary_df['Porcentagem (%)'].round(2)


print("Resumo de Conversão por Grupo e Etapa:")
summary_df

Resumo de Conversão por Grupo e Etapa:


,Etapa,Grupo,Quantidade (Sucessos),Total (Tentativas),Porcentagem (%)
0,product_page,A,1676,2604,64.36
1,product_cart,A,780,2604,29.95
2,purchase,A,822,2604,31.57
3,product_page,B,490,877,55.87
4,product_cart,B,243,877,27.71
5,purchase,B,246,877,28.05


À primeira vista, o Grupo B (o novo sistema de recomendação) está apresentando taxas de conversão piores do que o Grupo A em todas as etapas do funil.

#### Teste A/B

In [36]:
alpha = 0.05

In [37]:
def run_z_test(success_a, total_a, success_b, total_b, event_name, alpha):
    """
    Executa o teste Z de proporções e imprime um resumo claro dos resultados.
    """
    # Sucessos e tentativas para os grupos A e B
    count = [success_b, success_a]  # Ordem: Grupo Teste, Grupo Controle
    nobs = [total_b, total_a]

    # Executa o teste Z
    stat, p_value = proportions_ztest(count, nobs, alternative='two-sided')

    # Imprime os resultados
    print(f"--- Análise para a Etapa: '{event_name}' ---")
    print(f"Taxa de conversão Grupo A: {success_a / total_a:.2%}")
    print(f"Taxa de conversão Grupo B: {success_b / total_b:.2%}")
    print(f"P-valor: {p_value:.4f}")

    if p_value < alpha:
        print(f"Resultado: A diferença é estatisticamente significativa (p < {alpha:.4f}).")
    else:
        print(f"Resultado: Não há evidência de diferença estatisticamente significativa (p >= {alpha:.4f}).")
    print("-" * 40)
    print()


# Executando o teste para cada etapa do funil
# Etapa 1: product_page
run_z_test(success_page_a, total_a, success_page_b, total_b, 'product_page', alpha)

# Etapa 2: product_cart
run_z_test(success_cart_a, total_a, success_cart_b, total_b, 'product_cart', alpha)

# Etapa 3: purchase
run_z_test(success_purchase_a, total_a, success_purchase_b, total_b, 'purchase', alpha)

--- Análise para a Etapa: 'product_page' ---
Taxa de conversão Grupo A: 64.36%
Taxa de conversão Grupo B: 55.87%
P-valor: 0.0000
Resultado: A diferença é estatisticamente significativa (p < 0.0500).
----------------------------------------

--- Análise para a Etapa: 'product_cart' ---
Taxa de conversão Grupo A: 29.95%
Taxa de conversão Grupo B: 27.71%
P-valor: 0.2067
Resultado: Não há evidência de diferença estatisticamente significativa (p >= 0.0500).
----------------------------------------

--- Análise para a Etapa: 'purchase' ---
Taxa de conversão Grupo A: 31.57%
Taxa de conversão Grupo B: 28.05%
P-valor: 0.0508
Resultado: Não há evidência de diferença estatisticamente significativa (p >= 0.0500).
----------------------------------------



In [38]:
n_tests = 3  # Estamos testando 3 etapas
corrected_alpha = alpha / n_tests # Bonferroni

print(f"Nível de significância original (alpha): {alpha}")
print(f"Número de testes: {n_tests}")
print(f"Alpha corrigido com Bonferroni: {corrected_alpha:.4f}")

alpha = corrected_alpha

Nível de significância original (alpha): 0.05
Número de testes: 3
Alpha corrigido com Bonferroni: 0.0167


In [39]:

# Executando o teste para cada etapa do funil
# Etapa 1: product_page
run_z_test(success_page_a, total_a, success_page_b, total_b, 'product_page', alpha)

# Etapa 2: product_cart
run_z_test(success_cart_a, total_a, success_cart_b, total_b, 'product_cart', alpha)

# Etapa 3: purchase
run_z_test(success_purchase_a, total_a, success_purchase_b, total_b, 'purchase', alpha)

--- Análise para a Etapa: 'product_page' ---
Taxa de conversão Grupo A: 64.36%
Taxa de conversão Grupo B: 55.87%
P-valor: 0.0000
Resultado: A diferença é estatisticamente significativa (p < 0.0167).
----------------------------------------

--- Análise para a Etapa: 'product_cart' ---
Taxa de conversão Grupo A: 29.95%
Taxa de conversão Grupo B: 27.71%
P-valor: 0.2067
Resultado: Não há evidência de diferença estatisticamente significativa (p >= 0.0167).
----------------------------------------

--- Análise para a Etapa: 'purchase' ---
Taxa de conversão Grupo A: 31.57%
Taxa de conversão Grupo B: 28.05%
P-valor: 0.0508
Resultado: Não há evidência de diferença estatisticamente significativa (p >= 0.0167).
----------------------------------------



In [40]:
# Pivota o DataFrame para facilitar o cálculo da diferença
pivot_df = summary_df.pivot(index='Etapa', columns='Grupo', values='Porcentagem (%)')

# Calcula a mudança relativa do Grupo B em relação ao Grupo A
pivot_df['Mudanca_Relativa (%)'] = ((pivot_df['B'] / pivot_df['A']) - 1) * 100

# Exibe a mudança para cada etapa
print("Mudança Relativa na Conversão (B vs A):")
print(pivot_df.loc['product_page']['Mudanca_Relativa (%)'].round(1), '% para product_page')
print(pivot_df.loc['product_cart']['Mudanca_Relativa (%)'].round(1), '% para product_cart')
print(pivot_df.loc['purchase']['Mudanca_Relativa (%)'].round(1), '% para purchase')

Mudança Relativa na Conversão (B vs A):
-13.2 % para product_page
-7.5 % para product_cart
-11.1 % para purchase



### Interpretação dos Resultados

 **Falhas na Execução do Teste:** O teste A/B foi comprometido por falhas graves em sua execução, incluindo a coleta de dados da região errada, um número insuficiente de participantes e um grave desbalanceamento entre os grupos. Esses problemas limitam a validade e a generalização dos resultados.
    
- **Desempenho Inferior do Novo Sistema:** O Grupo B (novo sistema de recomendação) apresentou taxas de conversão **inferiores** ao Grupo A (controle) em todas as etapas analisadas.
    
- **Resultados do Teste Z (com correção de Bonferroni):**
    
    - **Página do Produto (product_page):** A queda na conversão do Grupo B foi **estatisticamente significativa**. O novo sistema prejudicou o engajamento inicial dos usuários.
        
    - **Carrinho (product_cart) e Compra (purchase):** As diferenças observadas não foram estatisticamente significativas após a correção. Não há evidências de que o novo sistema melhorou ou piorou a conversão nessas etapas finais.

- **Significância Estatística (com α corrigido = 0.0167):**
    
    - product_page: **Diferença significativa.** (p-valor ≈ 0.0000)
        
    - product_cart: **Diferença não significativa.** (p-valor ≈ 0.2067)
        
    - purchase: **Diferença não significativa.** (p-valor ≈ 0.0508)
        
- **Comparação com a Meta de 10% de Aumento:**  
    O novo sistema não apenas falhou em atingir a meta, como também resultou em uma queda de desempenho:
    
    - **Conversão para product_page:** Queda de **-13%**
        
    - **Conversão para product_cart:** Queda de **-7%**
        
    - **Conversão para purchase:** Queda de **-11%**

# Conclusão-Final

**O novo sistema de recomendação é ineficaz e prejudicial ao engajamento do usuário.** A análise, mesmo com os dados falhos, mostra uma queda estatisticamente significativa na primeira etapa do funil, sem nenhuma melhora nas etapas subsequentes.

#### Recomendações

1. **Não Implementar:** O novo sistema de recomendação **não deve ser implementado**. Os dados sugerem que ele impacta negativamente a experiência do usuário.
    
2. **Reexecutar o Teste:** Devido às graves falhas na execução do teste original, recomenda-se planejar e executar um novo teste A/B, garantindo que:
    
    - O público-alvo correto seja selecionado (região da UE).
        
    - O tamanho da amostra planejado seja alcançado.
        
    - O mecanismo de alocação de usuários divida os grupos de forma equilibrada (50/50).
        
    - O rastreamento de eventos seja consistente e confiável.
        
3. **Investigar a Causa da Queda:** Antes de um novo teste, a equipe de produto deve investigar por que o sistema de recomendação causou uma queda tão acentuada na visualização de páginas de produtos.

---
[Voltar ao Sumário ↑](#Sumário)
---

In [41]:
# import os

# # --- Preparação dos dados finais para o Tableau ---

# # 1. Criação do diretório de saída
# output_dir = 'dadostratados'
# os.makedirs(output_dir, exist_ok=True)

# # 2. Criação da tabela principal (Master Table) com todos os dados relevantes
# # Une os eventos filtrados com os participantes do teste e seus detalhes demográficos.
# df_tableau_master = df_events.merge(df_participants[['user_id', 'group']], on='user_id', how='inner')
# df_tableau_master = df_tableau_master.merge(df_new_users[['user_id', 'region', 'device', 'first_date']], on='user_id', how='left')

# # Seleciona e renomeia colunas para clareza no Tableau
# df_tableau_master = df_tableau_master[['user_id', 'group', 'region', 'device', 'first_date', 'event_dt', 'event_name', 'details']]
# df_tableau_master.columns = ['ID Usuario', 'Grupo', 'Regiao', 'Dispositivo', 'Data Cadastro', 'Data Evento', 'Nome Evento', 'Detalhes Evento']

# # 3. Preparação da tabela resumo para o gráfico de conversão por grupo
# # O dataframe `summary_df` já foi criado anteriormente e contém os dados exatos.
# # Apenas renomeamos as colunas para o português.
# summary_df_tableau = summary_df.rename(columns={
#     'Etapa': 'Etapa Funil',
#     'Grupo': 'Grupo',
#     'Quantidade (Sucessos)': 'Usuarios Convertidos',
#     'Total (Tentativas)': 'Total de Usuarios',
#     'Porcentagem (%)': 'Taxa de Conversao (%)'
# })


# # 4. Salvando os arquivos .csv no diretório criado
# df_tableau_master.to_csv(os.path.join(output_dir, 'dados_eventos_completos.csv'), index=False)
# summary_df_tableau.to_csv(os.path.join(output_dir, 'resumo_conversao_grupos.csv'), index=False)